# GörEndir — YouTube Video & Subtitle Downloader

This notebook demonstrates how to use **GörEndir** on Google Colab with Google Drive storage.

## Features
- Download single videos or entire playlists
- Multi-language subtitle extraction (SRT + TXT)
- `Dict[str, int]` input format: `{"url": start_number}` to skip and number from a specific video
- `reverse_download=True`: reverse playlist order (last video → #1)
- `skip_download=True`: download only subtitles, skip video file
- `playlist_end=N`: limit number of videos to download
- tqdm progress bar for every download
- Automatic VTT → clean SRT conversion

In [ ]:
# Install GörEndir
!pip install -q --upgrade --force-reinstall git+https://github.com/MammadTavakoli/gorendir.git

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import GörEndir
from gorendir import YouTubeDownloader
# Alternative: from gorendir.downloader import YouTubeDownloader

In [ ]:
# Define the save location (Google Drive or local Colab storage)
save_path = "/content/drive/MyDrive/YouTube/"

## Helper Function

The `download_videos` function wraps `YouTubeDownloader.download_video()`.

### Input Format for `video_urls`

The `video_urls` parameter supports three formats:

| Format | Example | Description |
|--------|---------|-------------|
| `str` | `"https://youtube.com/watch?v=XXX"` | Single URL |
| `Dict[str, int]` | `{"https://youtube.com/playlist?list=XXX": 8}` | URL with start number |
| `List[Union[str, Dict[str, int]]]` | `["url1", {"url2": 5}]` | Mixed list |

When using `Dict[str, int]`, the integer value means:
- **Skip** the first N-1 videos in download order
- **Number** files starting from that number

In [ ]:
def download_videos(video_urls, save_path, reverse_download=False, skip_download=False, playlist_end=0):
    """
    Download videos using GörEndir.

    Args:
        video_urls: URL(s) to download. Can be:
            - str: Single URL
            - Dict[str, int]: {"url": start_number}  e.g. {"https://...playlist?list=XXX": 8}
            - List[Union[str, Dict[str, int]]]: Mixed list of URLs and URL-start_number dicts
        save_path: Directory to save downloads
        reverse_download: If True, reverse playlist order (last video -> #1)
        skip_download: If True, only download subtitles (skip video file)
        playlist_end: If > 0, stop after downloading this many videos
    """
    downloader = YouTubeDownloader(
        save_directory=save_path,
        max_resolution=1080,
        subtitle_languages=["en", "fa"],
    )

    downloader.download_video(
        video_urls=video_urls,
        reverse_download=reverse_download,
        skip_download=skip_download,
        force_download=False,
        yt_dlp_write_subs=True,
        download_subtitles=True,
        playlist_end=playlist_end,
    )

## Download Configurations

Define different download scenarios. Each config has:
- `name`: A label for display
- `reverse_download`: Whether to reverse playlist order
- `skip_download`: Whether to skip video (only download subtitles)
- `playlist_end`: Limit number of videos (0 = no limit)
- `urls`: The URL(s) - can be a **list of strings**, a **dict of url-start_number**, or a **mixed list**

In [ ]:
# -- Subtitle-only (reverse order) --
# Download only subtitles from playlists in reverse order
# Dict format: {"url": start_number} -- skip first N-1 videos, number from N
sbtitle_reverse_urls = {
    'name': 'sbtitle_reverse_urls',
    'reverse_download': True,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # Example: skip first 7 videos, start numbering from 8 (in reverse download order)
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 8},
    ]
}

In [ ]:
# -- Video + Subtitle (reverse order) --
# Download videos with subtitles from playlists in reverse order
video_reverse_urls = {
    'name': 'video_reverse_urls',
    'reverse_download': True,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
    ]
}

In [ ]:
# -- Subtitle-only (normal order) --
# Download only subtitles from playlists in normal (first->last) order
sbtitle_urls = {
    'name': 'sbtitle_urls',
    'reverse_download': False,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 3},  # start from video #3
    ]
}

In [ ]:
# -- Video + Subtitle (normal order) --
# Download videos with subtitles in normal (first->last) order
video_urls_config = {
    'name': 'video_urls',
    'reverse_download': False,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/watch?v=YOUR_VIDEO_ID",
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 5},  # start from video #5
    ]
}

## Run Downloads

Iterate over all configurations and process each one.

The `urls` list is passed directly to `download_video()`. GörEndir automatically handles:
- `str` items -> single URL
- `dict` items -> `Dict[str, int]` format with start number

In [ ]:
video_list = [sbtitle_reverse_urls, video_reverse_urls, sbtitle_urls, video_urls_config]

for config in video_list:
    print('*' * 20, config['name'], '*' * 20)
    urls = config['urls']
    if not urls:
        print(f"  No URLs configured for {config['name']}, skipping.")
        continue

    download_videos(
        video_urls=urls,
        save_path=save_path,
        reverse_download=config['reverse_download'],
        skip_download=config['skip_download'],
        playlist_end=config.get('playlist_end', 0),
    )

---

## Advanced Usage Examples

Below are additional examples showing different input formats.

In [ ]:
# Example 1: Single video URL as string
# download_videos("https://youtube.com/watch?v=dQw4w9WgXcQ", save_path)

In [ ]:
# Example 2: Dict format -- skip first 7 videos, start numbering from 8
# download_videos({"https://youtube.com/playlist?list=PLxxxxxxx": 8}, save_path)

In [ ]:
# Example 3: Mixed list -- some URLs with start number, some without
# download_videos([
#     "https://youtube.com/watch?v=abc123",                # single video, no start number
#     {"https://youtube.com/playlist?list=PLxxxx": 5},     # playlist, start from #5
#     {"https://youtube.com/playlist?list=PLyyyy": 1},     # playlist, from beginning
# ], save_path)

In [ ]:
# Example 4: Reverse + start number
# In reverse mode, the LAST video of the playlist becomes #1.
# With start_num=8, the first 7 videos (in reverse order) are skipped,
# and numbering starts from 8.
# download_videos(
#     {"https://youtube.com/playlist?list=PLxxxx": 8},
#     save_path,
#     reverse_download=True
# )

In [ ]:
# Example 5: Limit playlist to first 10 videos (or last 10 in reverse mode)
# download_videos(
#     "https://youtube.com/playlist?list=PLxxxx",
#     save_path,
#     playlist_end=10
# )

In [ ]:
# Example 6: Using YouTubeDownloader directly with all options
# downloader = YouTubeDownloader(
#     save_directory=save_path,
#     max_resolution=720,                    # limit to 720p
#     subtitle_languages=["en", "fa", "tr"], # subtitle languages
#     retry_attempts=5,                      # more retries
#     cookies_path="/content/cookies.txt",    # for age-restricted content
# )
#
# downloader.download_video(
#     video_urls={"https://youtube.com/playlist?list=PLxxxx": 3},
#     reverse_download=False,
#     skip_download=False,
#     force_download=True,           # re-download even if already downloaded
#     yt_dlp_write_subs=True,        # let yt-dlp also write subtitles
#     download_subtitles=True,        # download subtitles via YouTube API
#     playlist_end=0,                 # 0 = no limit
# )